# Distillation Debug Dump Viewer\n\nLoad `.pt` update dumps, decode teacher/student views, and recompute distillation terms.

In [ ]:
from __future__ import annotations\n\nimport glob\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\nfrom transformers import AutoTokenizer\n\nDUMP_GLOB = '/home/anhvth8/projects/slime/outputs/**/distill_debug_dumps/distill_debug_*.pt'\nTOKENIZER_PATH = '/home/anhvth8/ckpt/hf_models/Qwen/Qwen3-4B-As-Qwen35'  # change as needed\n

In [ ]:
paths = sorted(glob.glob(DUMP_GLOB, recursive=True))\nprint(f'found dumps: {len(paths)}')\nfor p in paths[-5:]:\n    print(p)

In [ ]:
def load_payload(path: str):\n    payload = torch.load(path, map_location='cpu', weights_only=False)\n    print('path:', path)\n    print('mode:', payload.get('mode'))\n    print('rollout_id:', payload.get('rollout_id'))\n    print('records:', payload.get('num_records_kept'), '/', payload.get('num_records_total'))\n    print('recipe:', payload.get('recipe'))\n    print('limits:', payload.get('limits'))\n    return payload\n\npayload = load_payload(paths[-1]) if paths else None

In [ ]:
tok = AutoTokenizer.from_pretrained(TOKENIZER_PATH, trust_remote_code=True)\n\ndef decode_ids(ids):\n    if isinstance(ids, torch.Tensor):\n        ids = ids.tolist()\n    return tok.decode(ids, skip_special_tokens=False)\n\ndef show_views(record):\n    student_ids = record['student_input_ids']\n    teacher_ids = record['teacher_input_ids']\n    rs = int(record['response_start'])\n    rl = int(record['response_length'])\n    te_start = int(record['teacher_logprob_start_len'])\n\n    print('sample_index:', record.get('sample_index'))\n    print('response_start:', rs, 'response_length:', rl)\n    print('teacher_logprob_start_len:', te_start)\n    print('--- student prompt ---')\n    print(decode_ids(student_ids[:rs]))\n    print('--- student response ---')\n    print(decode_ids(student_ids[rs:rs+rl]))\n    print('--- teacher prompt side ---')\n    print(decode_ids(teacher_ids[:te_start]))\n    print('--- teacher response side ---')\n    print(decode_ids(teacher_ids[te_start:te_start+rl]))

In [ ]:
record_idx = 0\nif payload is not None and payload.get('records'):\n    rec = payload['records'][record_idx]\n    show_views(rec)

In [ ]:
def summarize_topk_record(record):\n    pos = record['position_indices'].tolist()\n    ids = record['teacher_topk_token_ids']\n    t_lp = record['teacher_topk_logprobs']\n    s_lp = record['student_topk_logprobs']\n    rows = []\n    for i, p in enumerate(pos):\n        rows.append({\n            'position': p,\n            'teacher_top1_id': int(ids[i, 0]),\n            'teacher_top1_lp': float(t_lp[i, 0]),\n            'student_on_teacher_top1_lp': float(s_lp[i, 0]),\n            'forward_kl': float(record['forward_kl'][i]),\n            'reverse_kl': float(record['reverse_kl'][i]),\n            'jsd': None if record.get('jsd') is None else float(record['jsd'][i]),\n        })\n    return pd.DataFrame(rows)\n\nif payload is not None and payload.get('mode') in {'fkl', 'mixed', 'jsd'}:\n    display(summarize_topk_record(payload['records'][record_idx]))

In [ ]:
def recompute_terms(record, mode: str, recipe: dict):\n    if mode == 'rkl':\n        rev = record['student_log_probs'] - record['teacher_log_probs']\n        return {'reverse_kl_mean': float(rev.mean()), 'logged_reverse_kl_mean': float(record['reverse_kl'].mean())}\n\n    teacher = record['teacher_topk_logprobs']\n    student = record['student_topk_logprobs']\n    t_norm = teacher - torch.logsumexp(teacher, dim=-1, keepdim=True)\n    s_norm = student - torch.logsumexp(student, dim=-1, keepdim=True)\n    t_prob = t_norm.exp()\n    s_prob = s_norm.exp()\n    fkl = (t_prob * (t_norm - s_norm)).sum(dim=-1)\n    rkl = (s_prob * (s_norm - t_norm)).sum(dim=-1)\n\n    out = {'fkl_mean': float(fkl.mean()), 'rkl_mean': float(rkl.mean())}\n    beta = float(recipe.get('opd_jsd_beta', 0.5))\n    m = beta * t_prob + (1.0 - beta) * s_prob\n    log_m = m.clamp_min(1e-12).log()\n    jsd = beta * (t_prob * (t_norm - log_m)).sum(dim=-1) + (1.0 - beta) * (s_prob * (s_norm - log_m)).sum(dim=-1)\n    out['jsd_mean'] = float(jsd.mean())\n    out['logged_forward_kl_mean'] = float(record['forward_kl'].mean())\n    out['logged_reverse_kl_mean'] = float(record['reverse_kl'].mean())\n    if record.get('jsd') is not None:\n        out['logged_jsd_mean'] = float(record['jsd'].mean())\n    return out\n\nif payload is not None and payload.get('records'):\n    print(recompute_terms(payload['records'][record_idx], payload['mode'], payload.get('recipe', {})))